<a href="https://colab.research.google.com/github/netsetos/agentic-ai-weekend-gcp-learners/blob/rag-production-hardening/module-12-production-deploy/lesson-12.3-admin-observability/notebooks/GCP_Capstone_12.3_AdminObservability.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 12.3 Admin & Observability — The Row Is the Dashboard
**Netsetos GenAI Engineering — GCP Capstone** · Module 12 · rebuilt on the live lane, 10 September 2026

The operability layer every regulated tenant demands - and what it looks like on the lane that exists. The API leaves one row per answer in Cloud Logging; a sink copies it into BigQuery, `tenant_daily` groups it by day and tenant, and the admin service renders that view - the same row, twice, minutes apart. This notebook reads the row where it lands first, the two log-based metrics and the alert policies from the project, groups the rows per tenant in rupees with the same GROUP BY the view runs, lists the retention-locked audit trail, runs the one DLP list the worker scans with, and reads the warehouse half - the sink, the view, the scan, the admin service - from the clone with the bill beside it. The fourteen heredocs at the end are the kit's source, unchanged.


## Setup


In [ ]:
!pip install -q google-genai==2.22.0 google-cloud-storage==3.13.1 google-cloud-firestore==2.30.0 google-cloud-dlp==3.39.0 pandas==2.3.3 requests==2.34.2

from google.colab import auth
auth.authenticate_user()

PROJECT_ID = "documind-ai-YOUR-ID"   # CHANGE THIS: the project the lane runs in (make up, lesson 4.8)
REGION     = "us-central1"
TENANT     = "acme"
KIT        = "/content/agentic-ai-weekend-gcp-learners"   # the kit: deploy/shared is the tool layer every lesson on the lane imports
BRANCH     = "rag-production-hardening"        # the learner repo's branch: the notebooks and the kit (deploy/) ship there together

import os, subprocess, sys
import google.auth
from google.auth.transport.requests import AuthorizedSession
from google import genai
from google.genai import types

if not os.path.isdir(KIT):
    subprocess.run(["git", "clone", "--depth", "1", "-q", "-b", BRANCH,
                    "https://github.com/netsetos/agentic-ai-weekend-gcp-learners", KIT], check=True)
sys.path.insert(0, f"{KIT}/deploy")                   # `from shared import ...` - the same layer every service imports

# The lane's URLs are deterministic: service name + project NUMBER (eventarc.tf builds them the same way).
creds, _ = google.auth.default()
NUMBER = AuthorizedSession(creds).get(
    f"https://cloudresourcemanager.googleapis.com/v1/projects/{PROJECT_ID}").json()["projectNumber"]
API_URL       = f"https://documind-api-{NUMBER}.{REGION}.run.app"
UPLOAD_BUCKET = f"{PROJECT_ID}-uploads"     # storage.tf: the bucket eventarc.tf watches - the corpus, media included
MEDIA_BUCKET  = f"{PROJECT_ID}-media"       # storage.tf: generated assets, 30-day lifecycle (a cache, not a record)
DATASETS      = f"{PROJECT_ID}-datasets"    # storage.tf (Module 10): the frozen tuning dataset and 10.5's GGUF
GATEWAY_URL   = f"https://documind-gateway-{NUMBER}.{REGION}.run.app"   # 11.3: LiteLLM on Cloud Run, behind IAM (make deploy-gateway)
SLM_URL       = f"https://documind-slm-{NUMBER}.{REGION}.run.app"       # 11.4: Ollama on an L4, min-instances 0 (make deploy-slm)
os.environ.update({
    "GOOGLE_CLOUD_PROJECT": PROJECT_ID,
    "GOOGLE_CLOUD_LOCATION": "global",              # Gemini 3.x generation is served from the global endpoint
    "GOOGLE_GENAI_USE_VERTEXAI": "TRUE",
    "DOCUMIND_PROFILE": "gcp",
    "RAG_API_URL": API_URL,
    "RAG_TIMEOUT_S": "90",                          # 7.2's finding: a cold API takes longer than the default 20 s
    # A notebook has no metadata server to be anyone with: the kit mints its ID tokens AS this roster
    # member (7.1). On Cloud Run the service's own account is the identity and nothing is set.
    "DOCUMIND_IMPERSONATE_SA": f"documind-ui-sa@{PROJECT_ID}.iam.gserviceaccount.com",
})
MEMBER_SA   = os.environ["DOCUMIND_IMPERSONATE_SA"]
OUTSIDER_SA = f"documind-outsider-sa@{PROJECT_ID}.iam.gserviceaccount.com"   # IAM admits it, no roster does (4.8, 7.2)

from shared import documind_tools    # THE one retrieve(). Imported, never pasted - the contract gate fails a paste.
gen = genai.Client(enterprise=True, project=PROJECT_ID, location="global")   # every generate_content in this lesson
HOURS = 24   # the window every read below uses

print("kit:", KIT, "| API:", API_URL, "| gateway:", GATEWAY_URL, "| slm:", SLM_URL)


## Cell 1: The API, the gateway and the SLM, called the way the lane calls them


In [ ]:
import json, requests, time, subprocess, datetime
from google.cloud import storage

# THE API, THE GATEWAY AND THE SLM, CALLED THE WAY THE LANE CALLS THEM: one ID token per request, minted AS the roster
# member for the service's own URL (the kit mints it: documind_tools._id_token). Every service on the lane is behind
# Cloud Run IAM; there is no master key and no API key to paste.
def api(path: str, body: dict | None = None, base: str | None = None, timeout: int = 120, token: str | None = "member") -> tuple[int, dict | str]:
    """POST one API route (or a candidate revision's, with base=) - as documind-ui-sa by default, with token=None as
    nobody, or with a token minted as another account. Returns (status, json-or-text)."""
    url = (base or API_URL).rstrip("/")
    headers = {}
    if token == "member":
        headers["Authorization"] = f"Bearer {documind_tools._id_token(API_URL)}"     # the audience is the canonical URL
    elif token:
        headers["Authorization"] = f"Bearer {token}"
    r = requests.post(f"{url}{path}", json=body, headers=headers, timeout=timeout)
    try:
        return r.status_code, r.json()
    except ValueError:
        return r.status_code, r.text[:400]

def api_get(path: str, base: str | None = None, timeout: int = 60) -> tuple[int, dict | str]:
    """GET one API route as the roster member: /version, /health."""
    url = (base or API_URL).rstrip("/")
    r = requests.get(f"{url}{path}", headers={"Authorization": f"Bearer {documind_tools._id_token(API_URL)}"}, timeout=timeout)
    try:
        return r.status_code, r.json()
    except ValueError:
        return r.status_code, r.text[:400]

def gateway(model: str, content: str, json_mode: bool = False, system: str | None = None, max_tokens: int = 200,
            timeout: int = 150) -> tuple[int, dict | str, dict]:
    """One OpenAI-compatible completion through the gateway (11.3), as the roster member. Returns (status, body, headers)."""
    msgs = ([{"role": "system", "content": system}] if system else []) + [{"role": "user", "content": content}]
    body = {"model": model, "messages": msgs, "max_tokens": max_tokens}
    if json_mode:
        body["response_format"] = {"type": "json_object"}
    r = requests.post(f"{GATEWAY_URL}/v1/chat/completions", json=body, timeout=timeout,
                      headers={"Authorization": f"Bearer {documind_tools._id_token(GATEWAY_URL)}"})
    try:
        return r.status_code, r.json(), dict(r.headers)
    except ValueError:
        return r.status_code, r.text[:400], dict(r.headers)

def service(name: str, region: str | None = None) -> dict:
    """A Cloud Run service as deployed: its env, labels, traffic, image and floor - read with gcloud, the way 10.3 did."""
    r = subprocess.run(["gcloud", "run", "services", "describe", name, "--region", region or REGION, "--project", PROJECT_ID,
                        "--format=json"], capture_output=True, text=True)
    if r.returncode != 0:
        return {}
    j = json.loads(r.stdout)
    c = j["spec"]["template"]["spec"]["containers"][0]
    return {"env": {e["name"]: e.get("value", "") for e in c.get("env", [])}, "image": c.get("image"),
            "labels": (j["metadata"].get("labels") or {}), "traffic": j.get("status", {}).get("traffic", []),
            "url": j.get("status", {}).get("url"), "sa": j["spec"]["template"]["spec"].get("serviceAccountName"),
            "annotations": (j["spec"]["template"]["metadata"].get("annotations") or {}),
            "min_instances": (j["spec"]["template"]["metadata"].get("annotations") or {}).get("autoscaling.knative.dev/minScale", "0")}

# The usage rows the API logs - the ONE shape every observability consumer reads (12.3, tenant_daily). They land in
# Cloud Logging first (the sink copies them into BigQuery for the view); this reads the last few for a surface, newest first.
def usage_rows(minutes: int = 15, limit: int = 20, event: str = "query", service_name: str = "documind-api") -> list[dict]:
    since = (datetime.datetime.now(datetime.timezone.utc) - datetime.timedelta(minutes=minutes)).strftime("%Y-%m-%dT%H:%M:%SZ")
    r = subprocess.run(["gcloud", "logging", "read",
                        f'resource.type="cloud_run_revision" AND resource.labels.service_name="{service_name}" '
                        f'AND jsonPayload.event="{event}" AND timestamp>="{since}"',
                        "--project", PROJECT_ID, "--limit", str(limit), "--format=json"], capture_output=True, text=True)
    try:
        return [e["jsonPayload"] for e in json.loads(r.stdout or "[]")]
    except ValueError:
        return []

# THE TWENTY LINES THAT MATTER, FROM THE CLONE. Module 12's notebooks are where the kit's files come from (the heredoc
# cells at the end of each notebook are what extract_documind.py reads), so the walls stay there and the story reads
# the file the lane actually runs, around one line, with the file's length beside it.
def excerpt(rel: str, needle: str, before: int = 0, after: int = 14) -> str:
    lines = open(f"{KIT}/deploy/{rel}", encoding="utf-8").read().splitlines()
    i = next(n for n, l in enumerate(lines) if needle in l)
    lo, hi = max(0, i - before), min(len(lines), i + after)
    return f"# {rel}:{lo + 1}-{hi}  ({len(lines)} lines)\n" + "\n".join(lines[lo:hi])

def gcloud(*args: str) -> str:
    """One gcloud read, as the notebook's account, stdout only."""
    r = subprocess.run(["gcloud", *args, "--project", PROJECT_ID], capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else f"(gcloud: {r.stderr.strip()[:200]})"

gcs = storage.Client(project=PROJECT_ID)

sys.path.insert(0, f"{KIT}/deploy/evals")               # run_eval, usage_rows, judge
sys.path.insert(0, f"{KIT}/deploy/services/rag-api")    # the API's own modules, for the excerpts and the pure functions
print("helpers: api(), api_get(), gateway(), service(), usage_rows(), excerpt(), gcloud(); the kit's evals/ and rag-api/ on sys.path")


## Cell 2: The row
The one shape every observability consumer reads, from Cloud Logging.


In [ ]:
# THE ROW. Every answer the API gives leaves one JSON object in Cloud Logging (main.py: usage_row - the ONE shape every
# consumer reads): who asked, which tenant, which model answered and through which door, tokens, cost priced at that
# model's rate, latency, the brain, and since 12.6 what the guard said. tenant_daily.sql selects exactly these fields.
# Read from Logging here, where the API writes them - minutes before the sink has copied them into BigQuery. Same row, other store.
rows = usage_rows(minutes=HOURS * 60, limit=300)
print(f"{len(rows)} query rows in the last {HOURS} h")
if rows:
    r0 = rows[0]
    for k in ("event", "tenant", "user", "model", "model_backend", "guard", "tokens_in", "tokens_out", "cached_tokens", "cost_usd",
              "latency_ms", "answerable", "unanswerable_flag", "confidence", "brain", "surface", "modality", "prompt_version", "retrieval_mode"):
        print(f"  {k:17} {r0.get(k)!r}")
    assert {"tenant", "model", "model_backend", "cost_usd", "latency_ms", "brain"} <= set(r0), "the row's shape is the contract"
else:
    print("  no rows yet: ask a question first (12.2's Cell 2, or make smoke) and re-run")


## Cell 3: The metrics and the alarms


In [ ]:
# THE METRICS AND THE ALARMS. Two log-based metrics count the rows (queries) and the rows whose unanswerable_flag is 1
# (alerts.tf) - the flag is a 0/1 beside the boolean because a log filter cannot turn true/false into a number - and a
# policy alerts on their ratio: a product signal an infrastructure dashboard cannot see. Beside it the latency SLO and,
# since Module 11's evening, the two "left warm" alarms. Read from the project through the APIs the console uses.
sess = AuthorizedSession(creds)
mets = sess.get(f"https://logging.googleapis.com/v2/projects/{PROJECT_ID}/metrics", timeout=60).json().get("metrics", [])
for m in mets:
    print(f"  metric  {m['name']:28} {m.get('filter', '')[:100]}")
assert {"documind/unanswerable", "documind/queries"} <= {m["name"] for m in mets}, "alerts.tf's two log-based metrics"
# The ledger's four (12 September 2026, deploy/INDEXING.md): the ingest events by name, what a reindex cost (embedded and
# reused, as distributions), and the drift the nightly job measures - read off the worker's and the job's log lines.
ledger_metrics = {"documind/ingest_events", "documind/ingest_embedded", "documind/ingest_reused", "documind/reconcile_drift"} & {m["name"] for m in mets}
print("the ledger's metrics:", sorted(ledger_metrics) or "not applied yet (alerts.tf: terraform apply is the owner's hour)")
pols = sess.get(f"https://monitoring.googleapis.com/v3/projects/{PROJECT_ID}/alertPolicies", timeout=60).json().get("alertPolicies", [])
for pol in pols:
    cond = (pol.get("conditions") or [{}])[0].get("conditionThreshold", {})
    print(f"  policy  {pol['displayName']:52} enabled={pol.get('enabled')} for {cond.get('duration')} -> {len(pol.get('notificationChannels', []))} channel(s)")
assert any("Unanswerable" in pol["displayName"] for pol in pols), "the product-signal alert"
print("the ledger's pagers:", [p_["displayName"] for p_ in pols if p_["displayName"] in ("Ingest failed", "Ledger drift above zero for two nights", "Nightly reconcile failed")]
      or "not applied yet - a failure within ten minutes, drift above zero for two nights, the job failing")
print("\nwhat fires it: questions the corpus cannot answer, refused - the golden set's refusal rows, asked on purpose - not an error rate")


## Cell 4: Per-tenant INR, from the rows
tenant_daily's GROUP BY, over the rows in Logging - the same figures the view holds once the sink has caught up.


In [ ]:
import pandas as pd
sys.path.insert(0, f"{KIT}/deploy/evals")
from usage_rows import read_rows, group

# PER-TENANT INR, FROM THE ROWS. tenant_daily is a BigQuery view: GROUP BY day, tenant, surface, backend, prompt version,
# retrieval mode, retrieval backend (the store that served), modality - a day late, once the sink has copied the rows. The
# same GROUP BY runs over the rows in Logging now (evals/usage_rows.py; make usage). The rupee column is what finance asked for, with the rate beside it,
# priced at the model that answered - a gateway answer at the gateway's own price.
data = read_rows(PROJECT_ID, HOURS)
print(f"{len(data)} rows (query, stream, media) in the last {HOURS} h")
if data:
    print(pd.DataFrame(group(data, ("tenant",))).to_string(index=False)); print()
    print(pd.DataFrame(group(data, ("model", "model_backend"))).to_string(index=False)); print()
    print(pd.DataFrame(group(data, ("brain",))).to_string(index=False)); print()
    print(pd.DataFrame(group(data, ("retrieval_backend",))).to_string(index=False))   # which store served the pool (13 September 2026)
print("\ntenant_daily, the same GROUP BY in BigQuery:")
print(excerpt("terraform/sql/tenant_daily.sql", "GROUP BY", 3, 3))


## Cell 5: The audit trail


In [ ]:
from shared.audit_log import AUDIT_ACTIONS

# THE AUDIT TRAIL. shared/audit_log.py writes one JSON object per sensitive action into the retention-locked audit
# bucket - the record, not a cache - and refuses an action name it does not know (adopting 9.4's media routes meant
# registering media.generate FIRST). Listed for the last events on record; the media generations of Module 9 are the
# ones a demo leaves. The Firestore hot index the admin dashboard reads is a cache; the bucket is the truth,
# and nothing can delete from it for five years.
print(sorted(AUDIT_ACTIONS))
bucket = gcs.get_bucket(f"{PROJECT_ID}-audit")
print(f"retention {bucket.retention_period} s, locked={bucket.retention_policy_locked}")
events = sorted(bucket.list_blobs(max_results=2000), key=lambda b: b.updated or datetime.datetime.min.replace(tzinfo=datetime.timezone.utc), reverse=True)
print(f"{len(events)} events on record; the newest:")
for b in events[:5]:
    ev = json.loads(b.download_as_text())
    print(f"  {b.name:58} {ev.get('action'):16} {(ev.get('actor') or {}).get('email', '')}")
if events:
    assert json.loads(events[0].download_as_text()).get("action") in AUDIT_ACTIONS


## Cell 6: DLP, the one list


In [ ]:
from shared import pii

# DLP, THE ONE LIST THE LANE SCANS WITH. shared/pii.py holds the India info types (Aadhaar, PAN, GST among them) and the
# regions - text in asia-south1, pixels in asia-southeast1, because Mumbai offers no image inspection and the first live
# figure said so. The worker scans every chunk and every figure with it before indexing; the admin service's dlp.py
# wraps the same module. Two lists that drift is the worst kind of compliance bug - the scan misses a type, the
# dashboard reports zero findings, and both look correct - so there is one list, and this cell runs it.
print(pii.INDIA_INFO_TYPES, "| text in", pii.LOCATION, "| images in", pii.IMAGE_LOCATION)
findings = pii.inspect("Contact priya@acme.example, PAN ABCPE1234F, Aadhaar 2345 6789 0123 - about the notice period.")
for f in findings:
    print("  ", f)
assert findings, "a PAN, an Aadhaar and an e-mail in one sentence must produce findings"


## Cell 7: The warehouse half, from the clone


In [ ]:
# THE WAREHOUSE HALF. The sink that copies the rows into BigQuery, the dataset and the daily view, 5.5's feature tables and
# the Dataplex scan, and the admin service that renders them - declared unconditionally in the files this lesson owns since
# 15 September 2026 (they were the full profile's, behind `count = local.full ? 1 : 0`, and the demo lane went without).
# Read from the clone, with the price beside it: a few dollars a month for the dataset and the sink at this volume, the
# scan on top, and one more IAP surface.
for rel, needle in (("terraform/sink.tf", "google_logging_project_sink"), ("terraform/dataplex.tf", "google_dataplex_datascan"),
                    ("services/admin/admin_dashboard.py", "def tenant_daily(")):
    print(excerpt(rel, needle, 1, 8)); print()
gates = {f: open(f"{KIT}/deploy/terraform/{f}", encoding="utf-8").read().count("local.full") for f in ("sink.tf", "dataplex.tf")}
assert not any(gates.values()), f"a profile gate survives: {gates}"
print("count gates:", gates, "| documind-admin deployed:", bool(service("documind-admin")))
assert service("documind-admin"), "the admin service is one of the seven make up deploys (SERVICES); an older lane: make build deploy-services SERVICES=admin SCRIPTS=commands/lesson-12.3.sh"
print("make bq-views  (the view, run by make up)  |  make features  (the feature job, then the scan)  |  make usage  (the same GROUP BY from Logging, now)")


## Where this goes
- **12.6** adds the guard's verdict to the row and prices every answer in both currencies; **10.3** is where the breakers read the month's counter these rows fill.
- **12.8**'s full smoke is what turns a green dashboard into a proven one: it asks for refusals, not only answers.

## ✅ Lesson 12.3 complete
- ✅ The usage row read from Cloud Logging, field by field
- ✅ The two log-based metrics and the alert policies read from the project; what fires the product signal
- ✅ Per-tenant, per-model and per-brain rupees as tenant_daily's GROUP BY over the rows
- ✅ The audit trail listed from the retention-locked bucket; the registered actions
- ✅ The one DLP list run live; the sink, the view, the scan and the admin service read from the clone, the admin service found deployed


## The files this lesson owns
Below are the fourteen heredocs the extractor places: the log sink and the daily view (BigQuery), 5.5's feature tables and scan (Dataplex), the admin service's DLP and audit wrappers, its dashboard, entrypoint, identity, requirements and Dockerfile, the alert policies (readopted twice in Module 11's evening, for the e-mail channel and the left-warm alarms), the admin deploy and its smoke. Unchanged by the rebuild; the story above reads them from the clone.


In [ ]:
SINK_TF = '''
resource "google_bigquery_dataset" "observability" {
  dataset_id                  = "documind_observability"
  location                    = var.india_region
  description                 = "API structured logs, DLP findings, tenant rollups"
  default_table_expiration_ms = 7776000000 # 90 days for raw logs
  delete_contents_on_destroy  = false
}

resource "google_logging_project_sink" "api_to_bq" {
  name                   = "documind-api-to-bq"
  destination            = "bigquery.googleapis.com/projects/${var.project_id}/datasets/${google_bigquery_dataset.observability.dataset_id}"
  filter                 = <<EOT
    resource.type = "cloud_run_revision"
    (resource.labels.service_name = "documind-api" OR resource.labels.service_name = "documind-chat")
    (jsonPayload.event = "query" OR jsonPayload.event = "stream" OR jsonPayload.event = "chat")
  EOT
  unique_writer_identity = true
  bigquery_options { use_partitioned_tables = true }
}

# Sink identity needs BQ data editor
resource "google_bigquery_dataset_iam_member" "sink_writer" {
  dataset_id = google_bigquery_dataset.observability.dataset_id
  role       = "roles/bigquery.dataEditor"
  member     = google_logging_project_sink.api_to_bq.writer_identity
}
'''
with open('sink.tf', 'w') as f: f.write(SINK_TF)
print('sink.tf written')
print()
print('Partitioned by day on timestamp. One row per /v1/query call.')
print('Columns auto-derived from jsonPayload: tenant, user, latency_ms, tokens_in/out, confidence, answerable')


In [ ]:
DAILY_ROLLUP_SQL = '''
CREATE OR REPLACE VIEW `documind_observability.tenant_daily` AS
SELECT
  DATE(timestamp, "Asia/Kolkata") AS day,
  jsonPayload.tenant AS tenant,
  -- The dimensions an answer can be sliced by. "It got worse last Tuesday" is
  -- unanswerable without knowing which prompt version and which retrieval mode
  -- served it.
  jsonPayload.surface AS surface,
  jsonPayload.model_backend AS model_backend,
  jsonPayload.prompt_version AS prompt_version,
  jsonPayload.retrieval_mode AS retrieval_mode,
  jsonPayload.modality AS modality,
  -- Which store served the pool (13 September 2026, evening): vector | firestore | rag_engine | vertex_search - the
  -- EFFECTIVE backend per request (a tenant's pin, or the kit's index after the data-region fallback), so cost per
  -- backend is this GROUP BY and not an estimate. The same typing rule as the stage clocks below: run make bq-views
  -- after an API that logs retrieval_backend, managed_chunks and policy_fallback has answered once.
  jsonPayload.retrieval_backend AS retrieval_backend,
  COUNT(*) AS queries,
  COUNTIF(jsonPayload.answerable = false) AS unanswerable,
  SUM(CAST(jsonPayload.tokens_in  AS INT64)) AS tokens_in,
  SUM(CAST(jsonPayload.tokens_out AS INT64)) AS tokens_out,
  APPROX_QUANTILES(CAST(jsonPayload.latency_ms AS INT64), 100)[OFFSET(50)] AS p50_ms,
  APPROX_QUANTILES(CAST(jsonPayload.latency_ms AS INT64), 100)[OFFSET(95)] AS p95_ms,
  APPROX_QUANTILES(CAST(jsonPayload.latency_ms AS INT64), 100)[OFFSET(99)] AS p99_ms,
  -- Where the time went. p95_ms moved: which stage moved it? The API row clocks each stage on its
  -- own (main.py stage()), so the answer is a column, not a trace hunt. pool is the candidates the
  -- reranker saw - the number TOP_K_RETRIEVE sets and evals/ablate.py decides - read here after it
  -- moves. BigQuery types a sink table's jsonPayload from the rows it has seen: run `make bq-views`
  -- after an API that logs these four has answered once, or the CREATE fails on "Field name
  -- retrieve_ms does not exist" - a missing field is a refusal, never a column of NULLs.
  APPROX_QUANTILES(CAST(jsonPayload.retrieve_ms AS INT64), 100)[OFFSET(95)] AS p95_retrieve_ms,
  APPROX_QUANTILES(CAST(jsonPayload.rerank_ms   AS INT64), 100)[OFFSET(95)] AS p95_rerank_ms,
  APPROX_QUANTILES(CAST(jsonPayload.generate_ms AS INT64), 100)[OFFSET(95)] AS p95_generate_ms,
  ROUND(AVG(CAST(jsonPayload.pool AS FLOAT64)), 1) AS avg_pool,
  -- The Ranking API stood in for by the retrieval order (retriever.rerank's fallback, 12 September 2026): a
  -- day with a number here served degraded answers, and that number is what pages someone.
  SUM(CAST(jsonPayload.rerank_fallback AS INT64)) AS rerank_fallbacks,
  -- The managed stores' share of the pools, and the questions a tenant's data_region sent to the kit's index instead
  -- of a managed store (main.py retrieval_backend_for): a day with policy_fallbacks is a policy working, not a fault.
  SUM(CAST(jsonPayload.managed_chunks AS INT64)) AS managed_chunks,
  SUM(CAST(jsonPayload.policy_fallback AS INT64)) AS policy_fallbacks,
  SUM(CAST(jsonPayload.cached_tokens AS INT64)) AS cached_tokens,
  ROUND(SUM(CAST(jsonPayload.cost_usd AS FLOAT64)), 4) AS cost_usd,
  ROUND(SUM(CAST(jsonPayload.cost_usd AS FLOAT64)) * 85, 2) AS cost_inr
FROM `documind_observability.run_googleapis_com_stdout`
-- BOTH surfaces. Filtering on "query" alone excluded every streamed answer,
-- and 12.4's UI streams - so the primary user path produced no rows and this
-- view stayed empty however many questions anyone asked. "media" is 9.4's
-- Studio (services/rag-api/media.py, gap G8): modality=image, cost_usd per
-- image, no tokens - so media spend per tenant is a GROUP BY, not an estimate.
WHERE jsonPayload.event IN ("query", "stream", "media")
GROUP BY day, tenant, surface, model_backend, prompt_version, retrieval_mode,
         retrieval_backend, modality;
'''
with open('tenant_daily.sql', 'w') as f: f.write(DAILY_ROLLUP_SQL)
print('tenant_daily.sql written')
print()
print('Materialised view refreshes automatically. Query cost for dashboard ~10MB scanned.')


In [ ]:
DATAPLEX_TF = r'''
# Module 5's production lane: the feature tables and the quality gate. Lesson 5.5, adopted by
# 12.3 (gap G9).
#
# 5.5 taught this against a synthetic fixture in a dataset called rag_data. The SAME dataset
# name is used here, on purpose: every SQL statement in 5.5 runs unchanged against production,
# and what fills chunk_source is no longer a fixture - services/ingest/indexer.py streams one
# row per chunk it indexes (BQ_CHUNK_TABLE), with the canonical names 2.3/4.x/rag-api use and
# the DLP verdict the worker already computed. So "the SQL lane reads DocuMind's real chunks"
# is literally what happens, and pii_flag needs no second scan and no BigQuery model.
#
# Terraform DECLARES the tables (schemas), sql/chunk_metadata.sql COMPUTES them (`make features`):
# cheap features -> MERGE into chunk_metadata -> ingest_events -> the index_feed by subtraction.
# Declaring index_feed here rather than CREATE OR REPLACE-ing it is what lets the Dataplex scan
# below exist before the first job has run.
#
# The product was renamed - Dataplex Universal Catalog became Knowledge Catalog on 10 April 2026
# - but the API, the provider resource and the IAM roles all still say dataplex, which is why
# everything below does.

resource "google_project_service" "dataplex" {
  service            = "dataplex.googleapis.com"
  disable_on_destroy = false
}

resource "google_bigquery_dataset" "rag_data" {
  dataset_id                 = "rag_data"
  location                   = var.india_region
  description                = "Chunk features, ingest events and the index feed (lesson 5.5); chunk_source is streamed by the ingest worker"
  delete_contents_on_destroy = true # a learner's throwaway project: make down must take it
}

# The worker writes; the admin surface already reads project-wide (sa.tf).
resource "google_bigquery_dataset_iam_member" "ingest_writer" {
  dataset_id = google_bigquery_dataset.rag_data.dataset_id
  role       = "roles/bigquery.dataEditor"
  member     = "serviceAccount:${google_service_account.ingest.email}"
}

resource "google_bigquery_table" "chunk_source" {
  dataset_id          = google_bigquery_dataset.rag_data.dataset_id
  table_id            = "chunk_source"
  deletion_protection = false
  description         = "One row per indexed chunk, streamed by services/ingest - the canonical document, plus the worker's DLP verdict"
  time_partitioning {
    type  = "DAY"
    field = "ingested_at"
  }
  clustering = ["tenant_id", "doc_type"]
  schema = jsonencode([
    { name = "chunk_id", type = "STRING", mode = "REQUIRED", description = "Firestore document id in chunks" },
    { name = "tenant_id", type = "STRING", mode = "REQUIRED" },
    { name = "text", type = "STRING", mode = "NULLABLE" },
    { name = "source_uri", type = "STRING", mode = "NULLABLE" },
    { name = "page_start", type = "INTEGER", mode = "NULLABLE" },
    { name = "page_end", type = "INTEGER", mode = "NULLABLE" },
    { name = "doc_type", type = "STRING", mode = "NULLABLE" },
    { name = "kind", type = "STRING", mode = "NULLABLE", description = "text | figure | table | segment" },
    { name = "heading_path", type = "STRING", mode = "NULLABLE" },
    { name = "last_revised_at", type = "DATE", mode = "NULLABLE" },
    { name = "pii_flag", type = "BOOLEAN", mode = "NULLABLE", description = "TRUE keeps the chunk OUT of the shared index" },
    { name = "ingested_at", type = "TIMESTAMP", mode = "NULLABLE" },
  ])
}

resource "google_bigquery_table" "chunk_metadata" {
  dataset_id          = google_bigquery_dataset.rag_data.dataset_id
  table_id            = "chunk_metadata"
  deletion_protection = false
  description         = "One row per chunk: the feature table and the join key back to the corpus (5.5). No text, no vectors."
  time_partitioning {
    type  = "DAY"
    field = "featured_at"
  }
  clustering = ["tenant_id", "doc_type"]
  schema = jsonencode([
    { name = "chunk_id", type = "STRING", mode = "REQUIRED" },
    { name = "tenant_id", type = "STRING", mode = "REQUIRED" },
    { name = "source_uri", type = "STRING", mode = "NULLABLE" },
    { name = "page_start", type = "INTEGER", mode = "NULLABLE" },
    { name = "page_end", type = "INTEGER", mode = "NULLABLE" },
    { name = "token_count", type = "INTEGER", mode = "NULLABLE", description = "Approximate: characters / 4" },
    { name = "heading_depth", type = "INTEGER", mode = "NULLABLE" },
    { name = "has_table", type = "BOOLEAN", mode = "NULLABLE" },
    { name = "has_figure", type = "BOOLEAN", mode = "NULLABLE" },
    { name = "language", type = "STRING", mode = "NULLABLE", description = "en | hi | mixed" },
    { name = "freshness_days", type = "INTEGER", mode = "NULLABLE" },
    { name = "doc_type", type = "STRING", mode = "NULLABLE" },
    { name = "doc_type_conf", type = "FLOAT", mode = "NULLABLE" },
    { name = "pii_flag", type = "BOOLEAN", mode = "NULLABLE" },
    { name = "featured_at", type = "TIMESTAMP", mode = "NULLABLE" },
  ])
}

resource "google_bigquery_table" "ingest_events" {
  dataset_id          = google_bigquery_dataset.rag_data.dataset_id
  table_id            = "ingest_events"
  deletion_protection = false
  description         = "Append-only: what changed and when. 'withheld' is the row an auditor asks about."
  time_partitioning {
    type  = "DAY"
    field = "occurred_at"
  }
  clustering = ["tenant_id", "event_type"]
  schema = jsonencode([
    { name = "event_id", type = "STRING", mode = "REQUIRED" },
    { name = "chunk_id", type = "STRING", mode = "REQUIRED" },
    { name = "tenant_id", type = "STRING", mode = "REQUIRED" },
    { name = "event_type", type = "STRING", mode = "REQUIRED", description = "created | updated | reindexed | withheld" },
    { name = "reason", type = "STRING", mode = "NULLABLE" },
    { name = "occurred_at", type = "TIMESTAMP", mode = "REQUIRED" },
  ])
}

resource "google_bigquery_table" "index_feed" {
  dataset_id          = google_bigquery_dataset.rag_data.dataset_id
  table_id            = "index_feed"
  deletion_protection = false
  description         = "What SHIPS to the index - chunk_metadata minus PII and minus fragments, by subtraction. The scan below gates on this table."
  clustering          = ["tenant_id", "doc_type"]
  schema = jsonencode([
    { name = "chunk_id", type = "STRING", mode = "REQUIRED" },
    { name = "tenant_id", type = "STRING", mode = "REQUIRED" },
    { name = "doc_type", type = "STRING", mode = "NULLABLE" },
    { name = "language", type = "STRING", mode = "NULLABLE" },
    { name = "token_count", type = "INTEGER", mode = "NULLABLE" },
    { name = "freshness_days", type = "INTEGER", mode = "NULLABLE" },
    { name = "heading_depth", type = "INTEGER", mode = "NULLABLE" },
    { name = "has_table", type = "BOOLEAN", mode = "NULLABLE" },
  ])
}

# The gate - 5.5 cell 9's three rules, on the table that ships. If they fail, the index does
# not get rebuilt. On demand: `make features` runs the job and then the scan, and reads the
# result the way 5.5's dq_gate() does.
resource "google_dataplex_datascan" "chunk_dq" {
  location     = var.india_region
  data_scan_id = "documind-chunk-dq"
  display_name = "DocuMind index feed quality"

  data {
    resource = "//bigquery.googleapis.com/projects/${var.project_id}/datasets/${google_bigquery_dataset.rag_data.dataset_id}/tables/${google_bigquery_table.index_feed.table_id}"
  }

  execution_spec {
    trigger {
      on_demand {}
    }
  }

  data_quality_spec {
    # 1. one row per chunk - a duplicate is a duplicated citation
    rules {
      column    = "chunk_id"
      dimension = "UNIQUENESS"
      uniqueness_expectation {}
    }
    # 2. below 32 tokens a chunk carries no answer; above 2,048 it will not survive the packer.
    #    On the feed this is a regression test that the subtraction in the job still happens.
    rules {
      column    = "token_count"
      dimension = "VALIDITY"
      threshold = 0.99
      range_expectation {
        min_value = "32"
        max_value = "2048"
      }
    }
    # 3. nothing flagged as PII may be in the feed. Passes when the statement returns no rows.
    rules {
      dimension = "VALIDITY"
      sql_assertion {
        sql_statement = "SELECT chunk_id FROM `${var.project_id}.rag_data.chunk_metadata` WHERE pii_flag AND chunk_id IN (SELECT chunk_id FROM `${var.project_id}.rag_data.index_feed`)"
      }
    }
  }

  depends_on = [google_project_service.dataplex]
}

output "chunk_dq_scan" { value = google_dataplex_datascan.chunk_dq.name }
'''

with open('dataplex.tf', 'w') as f: f.write(DATAPLEX_TF)
print('dataplex.tf:', len(DATAPLEX_TF.splitlines()), 'lines')


In [ ]:
CHUNK_METADATA_SQL = r'''
-- Module 5's production lane: the feature job. Lesson 5.5 steps 3, 6 and 7, adopted by 12.3
-- (gap G9). Run it with `make features` - `bq query --use_legacy_sql=false < this file` - after
-- terraform has declared the rag_data tables (dataplex.tf); then the Dataplex scan gates the
-- index rebuild. Every statement is a full recompute or a MERGE, so re-running is safe.
--
-- What is different from the notebook, and why:
--   * chunk_source is not a fixture. services/ingest/indexer.py streams one row per chunk it
--     indexes, with the canonical names (text, source_uri, page_start, doc_type) and the DLP
--     verdict it already computed. So doc_type and pii_flag come from the ingest, not from a
--     BigQuery remote model - the expensive features (5.5 steps 4-5) are free here.
--   * index_feed is TRUNCATE + INSERT, not CREATE OR REPLACE: terraform owns its schema so the
--     quality scan can exist before the first job has run.
--   * The regexes are RE2, as BigQuery speaks it. `[\x{0900}-\x{097F}]` is the Devanagari block.

-- ---------------------------------------------------------------- step 3: cheap features
CREATE OR REPLACE TABLE `rag_data.chunk_features_cheap` AS
SELECT
  chunk_id,
  tenant_id,
  source_uri,
  page_start,
  page_end,
  -- Approximate, deliberately: an exact count is an API call per chunk. ~4 characters per
  -- token for English; Hindi runs richer, so this UNDER-counts, which is the safe direction.
  CAST(CEIL(LENGTH(text) / 4) AS INT64)                            AS token_count,
  ARRAY_LENGTH(SPLIT(COALESCE(heading_path, ''), ' > ')) - 1       AS heading_depth,
  REGEXP_CONTAINS(text, r'(?i)\|.*\|.*\||\btable\s+\d')            AS has_table,
  REGEXP_CONTAINS(text, r'(?i)\bfigure\s+\d|\bfig\.\s*\d')
    OR kind IN ('figure', 'table')                                 AS has_figure,
  CASE
    WHEN REGEXP_CONTAINS(text, r'[\x{0900}-\x{097F}]')
     AND REGEXP_CONTAINS(text, r'[A-Za-z]{4,}')                    THEN 'mixed'
    WHEN REGEXP_CONTAINS(text, r'[\x{0900}-\x{097F}]')             THEN 'hi'
    ELSE 'en'
  END                                                              AS language,
  DATE_DIFF(CURRENT_DATE(), COALESCE(last_revised_at, DATE(ingested_at)), DAY) AS freshness_days,
  doc_type,
  pii_flag
FROM `rag_data.chunk_source`;

-- ---------------------------------------------------------------- step 6: the feature table
MERGE `rag_data.chunk_metadata` T
USING (
  SELECT chunk_id, tenant_id, source_uri, page_start, page_end,
         token_count, heading_depth, has_table, has_figure, language, freshness_days,
         doc_type, 1.0 AS doc_type_conf, pii_flag,
         CURRENT_TIMESTAMP() AS featured_at
  FROM `rag_data.chunk_features_cheap`
) S
ON T.chunk_id = S.chunk_id AND T.tenant_id = S.tenant_id
WHEN MATCHED THEN UPDATE SET
  source_uri = S.source_uri, page_start = S.page_start, page_end = S.page_end,
  token_count = S.token_count, heading_depth = S.heading_depth, has_table = S.has_table,
  has_figure = S.has_figure, language = S.language, freshness_days = S.freshness_days,
  doc_type = S.doc_type, doc_type_conf = S.doc_type_conf, pii_flag = S.pii_flag,
  featured_at = S.featured_at
WHEN NOT MATCHED THEN INSERT ROW;

-- The append-only log. 'withheld' is the row an auditor will ask about.
INSERT INTO `rag_data.ingest_events`
  (event_id, chunk_id, tenant_id, event_type, reason, occurred_at)
SELECT GENERATE_UUID(), chunk_id, tenant_id,
       IF(COALESCE(pii_flag, TRUE), 'withheld', 'reindexed'),
       IF(COALESCE(pii_flag, TRUE), 'pii_flag=true', 'features recomputed'),
       CURRENT_TIMESTAMP()
FROM `rag_data.chunk_metadata`;

-- ---------------------------------------------------------------- step 7: the feed, by subtraction
-- Every chunk missing from the feed was removed by a WHERE clause in ONE place, rather than by
-- a filter each retrieval path has to remember. The scan's rule 2 is the regression test that
-- this subtraction still happens.
TRUNCATE TABLE `rag_data.index_feed`;
INSERT INTO `rag_data.index_feed`
  (chunk_id, tenant_id, doc_type, language, token_count, freshness_days, heading_depth, has_table)
SELECT chunk_id, tenant_id, doc_type, language, token_count, freshness_days, heading_depth, has_table
FROM `rag_data.chunk_metadata`
WHERE NOT COALESCE(pii_flag, TRUE)      -- unknown counts as unsafe
  AND token_count >= 32;                -- too short to carry an answer
'''

with open('chunk_metadata.sql', 'w') as f: f.write(CHUNK_METADATA_SQL)
print('chunk_metadata.sql:', len(CHUNK_METADATA_SQL.splitlines()), 'lines')


In [ ]:
DLP_PY = '''\n"""Admin-side DLP: the redaction path, and a thin wrapper over the shared scanner.\n\nThe info-type list is NOT defined here. It lives in shared/pii.py, because the ingest\nworker scans with it too and two lists that drift are the worst kind of bug in a\ncompliance control: the scan misses a type, the dashboard reports zero findings, and\nboth look like they are working.\n"""\nimport os\n\nfrom google.cloud import dlp_v2, firestore\n\nfrom shared.pii import INDIA_INFO_TYPES, LOCATION, PROJECT, inspect\n\n_dlp = dlp_v2.DlpServiceClient()\n_fs = firestore.Client(project=PROJECT)\n\n\ndef inspect_and_log(chunk_id: str, tenant_id: str, text: str) -> dict:\n    """Scan one chunk and record the findings - never the matched values."""\n    findings = inspect(text)\n    if findings:\n        _fs.collection("dlp_findings").add({\n            "chunk_id": chunk_id, "tenant_id": tenant_id,\n            "findings": findings, "count": len(findings),\n            "scanned_at": firestore.SERVER_TIMESTAMP,\n        })\n    return {"has_pii": bool(findings),\n            "types": sorted({f["info_type"] for f in findings})}\n\n\ndef redact(text: str) -> str:\n    """Use BEFORE sending to an external provider. Replaces each finding with its type."""\n    resp = _dlp.deidentify_content(request={\n        "parent": f"projects/{PROJECT}/locations/{LOCATION}",\n        "inspect_config": {"info_types": INDIA_INFO_TYPES,\n                           "min_likelihood": "POSSIBLE"},\n        "deidentify_config": {"info_type_transformations": {\n            "transformations": [{\n                "primitive_transformation": {\n                    "replace_with_info_type_config": {}}}]}},\n        "item": {"value": text},\n    })\n    return resp.item.value    # "Send to [EMAIL_ADDRESS] by [DATE_OF_BIRTH]"\n'''
with open('dlp.py', 'w') as f: f.write(DLP_PY)
print('dlp.py written')
print()
print('inspect_and_log() used on every uploaded chunk during ingestion (Module 11 pipeline hook)')
print('redact() used on every prompt routed to non-India providers (LiteLLM pre-request hook)')


In [ ]:
AUDIT_PY = '''\n"""Admin-side audit: the shared emitter, plus the Firestore index the dashboard reads.\n\nemit() itself lives in shared/audit_log.py so the ingest worker writes the SAME event\nshape into the SAME retention-locked bucket. What is admin-specific is the fast index:\nthe dashboard filters events by tenant and day, and listing a GCS prefix to do that\nwould be slow enough that nobody would use the page.\n"""\nimport google.auth\nimport os\n\nfrom google.cloud import firestore\n\nfrom shared.audit_log import AUDIT_ACTIONS, emit as _emit\n\n_fs = firestore.Client(project=(os.environ.get("GOOGLE_CLOUD_PROJECT")\n                                or google.auth.default()[1]))\n\n\ndef emit(action: str, actor: dict, target: dict, meta: dict | None = None) -> str:\n    """Write the immutable event, then index it for the dashboard (14-day TTL)."""\n    event_id = _emit(action, actor, target, meta)\n    _fs.collection("audit_index").document(event_id).set({\n        "id": event_id, "action": action, "actor": actor,\n        "target": target, "meta": meta or {},\n        "ts": firestore.SERVER_TIMESTAMP,\n    })\n    return event_id\n'''
with open('audit.py', 'w') as f: f.write(AUDIT_PY)
print('audit.py written')
print()
print('Dual-write pattern: GCS is source of truth (5y retention, locked).')
print('Firestore audit_index is a 14-day hot index for the dashboard.')


In [ ]:
ADMIN_PY = '''
import os, pandas as pd, plotly.express as px, streamlit as st
from google.cloud import bigquery, firestore

_bq = bigquery.Client()
import google.auth
_fs = firestore.Client(project=(os.environ.get("GOOGLE_CLOUD_PROJECT") or google.auth.default()[1]))
DATASET = "documind_observability"

@st.cache_data(ttl=300)
def tenant_daily(tenant: str | None, days: int = 30) -> pd.DataFrame:
    where = "day >= DATE_SUB(CURRENT_DATE('Asia/Kolkata'), INTERVAL @days DAY)"
    params = [bigquery.ScalarQueryParameter("days", "INT64", days)]
    if tenant:
        where += " AND tenant = @t"
        params.append(bigquery.ScalarQueryParameter("t", "STRING", tenant))
    sql = f"SELECT * FROM `{DATASET}.tenant_daily` WHERE {where} ORDER BY day"
    return _bq.query(sql, job_config=bigquery.QueryJobConfig(query_parameters=params)).to_dataframe()

def usage_tab():
    st.subheader("Usage (last 30 days)")
    tenant = st.selectbox("Tenant filter", ["All"] + list_tenants()) or "All"
    df = tenant_daily(None if tenant == "All" else tenant)
    if df.empty: st.info("No queries in window."); return
    col1, col2, col3, col4 = st.columns(4)
    col1.metric("Total queries", f"{df['queries'].sum():,}")
    col2.metric("Tokens (M)", f"{(df['tokens_in'].sum()+df['tokens_out'].sum())/1e6:.2f}")
    col3.metric("p95 latency", f"{df['p95_ms'].median():.0f} ms")
    unans_pct = 100*df["unanswerable"].sum()/max(1, df["queries"].sum())
    col4.metric("Unanswerable", f"{unans_pct:.1f} %")
    st.plotly_chart(px.bar(df, x="day", y="queries", color="tenant",
                           title="Queries per day"))
    st.plotly_chart(px.line(df, x="day", y=["p50_ms","p95_ms","p99_ms"],
                            title="Latency percentiles (ms)"))

def tenants_tab():
    st.subheader("Tenants")
    rows = []
    for t in _fs.collection("tenants").stream():
        d = t.to_dict(); d["id"] = t.id; rows.append(d)
    df = pd.DataFrame(rows)
    st.dataframe(df, use_container_width=True)
    with st.expander("Create tenant"):
        tid = st.text_input("Tenant ID")
        tier = st.selectbox("Tier", ["free","pro","enterprise"])
        quota = st.number_input("Monthly budget (USD)", 10, 10000, 50)
        if st.button("Create") and tid:
            _fs.collection("tenants").document(tid).set({
                "tier": tier, "max_budget_usd": quota,
                "created_at": firestore.SERVER_TIMESTAMP,
            })
            from audit import emit
            emit("tenant.create", st.session_state.user,
                 {"type":"tenant","id":tid}, {"tier":tier,"budget":quota})
            st.success(f"Created {tid}"); st.rerun()

def audit_tab():
    st.subheader("Audit log (last 14 days)")
    action = st.selectbox("Action", ["all","user.login","doc.upload","doc.delete",
                                     "admin.rotate_key","tenant.suspend"])
    q = _fs.collection("audit_index").order_by("ts", direction="DESCENDING").limit(500)
    if action != "all": q = q.where("action", "==", action)
    rows = [d.to_dict() for d in q.stream()]
    st.dataframe(pd.DataFrame(rows), use_container_width=True)
    st.caption("Full 5-year audit is in GCS. Firestore index is hot 14 days only.")

def dlp_tab():
    st.subheader("DLP findings")
    q = _fs.collection("dlp_findings").order_by("scanned_at", direction="DESCENDING").limit(200)
    rows = [d.to_dict() for d in q.stream()]
    df = pd.DataFrame(rows)
    if df.empty: st.info("No findings."); return
    st.metric("Chunks with findings", len(df))
    flat = []
    for _, r in df.iterrows():
        for f in r["findings"]:
            flat.append({"tenant": r["tenant_id"], "type": f["info_type"],
                         "likelihood": f["likelihood"]})
    st.plotly_chart(px.histogram(pd.DataFrame(flat), x="type", color="likelihood",
                                 title="PII findings by type"))

def ingestion_tab():
    """The ledger, as the admin sees it (12 September 2026, deploy/INDEXING.md): every source's current version
    and what its last reindex cost - chunks reused by hash against chunks embedded - the rows retired, the date
    a document declares, and each tenant's corpus fingerprint (what the API's cache is keyed to). Read from
    sources/ and ledger/, which the ingest worker writes on every event; the drift the nightly reconcile measures
    is a log-based metric (alerts.tf) and pages on its own."""
    st.subheader("Ingestion - the ledger")
    rows = []
    for s in _fs.collection("sources").stream():
        d = s.to_dict() or {}
        at = d.get("indexed_at")
        rows.append({"tenant": d.get("tenant_id"), "document": (d.get("name") or "").split("/", 1)[-1],
                     "status": d.get("status"), "chunks": d.get("chunks"), "reused": d.get("reused"),
                     "embedded": d.get("embedded"), "retired": d.get("retired"),
                     "effective_from": d.get("effective_from") or "",
                     "embedding": f"{d.get('embedding_model') or '?'}@{d.get('embedding_version') or '?'}",
                     "indexed": at.strftime("%Y-%m-%d %H:%M") if hasattr(at, "strftime") else ""})
    if not rows:
        st.info("No sources yet: make ingest-corpus, or make backfill-current on a lane older than the ledger.")
        return
    df = pd.DataFrame(rows).sort_values(["tenant", "document"])
    col1, col2, col3, col4 = st.columns(4)
    col1.metric("Current versions", int((df["status"] == "indexed").sum()))
    col2.metric("Retired sources", int((df["status"] == "retired").sum()))
    col3.metric("Chunks reused (last reindexes)", int(df["reused"].fillna(0).sum()))
    col4.metric("Chunks embedded (last reindexes)", int(df["embedded"].fillna(0).sum()))
    st.dataframe(df, use_container_width=True)
    for l in _fs.collection("ledger").stream():
        d = l.to_dict() or {}
        st.caption(f"{l.id}: corpus fingerprint {d.get('fingerprint')} over {d.get('versions')} current versions "
                   f"(last event {d.get('last_event')})")
    recent = df[df["reused"].notna()]
    if not recent.empty:
        st.plotly_chart(px.bar(recent, x="document", y=["reused", "embedded"], color="tenant", barmode="group",
                               title="What the last reindex of each source cost: reused by hash vs embedded"))


def list_tenants() -> list[str]:
    return sorted(t.id for t in _fs.collection("tenants").stream())

def admin_page(user):
    st.title("🛠 DocuMind Admin")
    tabs = st.tabs(["Usage", "Tenants", "Audit Log", "DLP",
                    "Ingestion", "Graph", "Cost", "Quality",
                    "Context & Memory"])
    with tabs[0]: usage_tab()
    with tabs[1]: tenants_tab()
    with tabs[2]: audit_tab()
    with tabs[3]: dlp_tab()
    with tabs[4]: ingestion_tab()
    # Stubs, deliberately visible. An empty tab that says which lesson fills it
    # is a roadmap; a tab that is missing entirely is a surprise in the demo.
    for i, (name, owner) in enumerate([
            ("Graph", "4.6 - entity/edge counts and orphaned nodes"),
            ("Cost", "12.6 - INR per tenant per day, from tenant_daily"),
            ("Quality", "10.4 - golden-set scores per prompt_version"),
            ("Context & Memory", "8.6 - store size and recall hit rate")], start=5):
        with tabs[i]:
            st.info(f"**{name}** lands in lesson {owner}.")
'''
with open('admin_dashboard.py', 'w') as f: f.write(ADMIN_PY)
print('admin_dashboard.py written')


In [ ]:
APP_PY = '''
"""documind-admin entrypoint.

Its own service, under admin-sa, so reporting credentials never live in the process
that renders user chat. The split is only real if the two also ship as two images.
"""
import streamlit as st

from admin_dashboard import admin_page
from auth import current_user

st.set_page_config(page_title="DocuMind Admin", page_icon="\U0001F6E0",
                   layout="wide")

user = current_user()          # raises/stops if IAP did not assert an admin
admin_page(user)
'''

AUTH_PY = '''
"""Who is asking, verified.

IAP puts a signed JWT in x-goog-iap-jwt-assertion. Reading x-goog-authenticated-user-email
without verifying the signature is the bug 12.2 removed from rag-api: a header is a claim,
not a credential, and anything that can reach the service can set one.
"""
import os

import streamlit as st
from google.auth.transport import requests as g_requests
from google.oauth2 import id_token

IAP_AUDIENCE = os.environ.get("IAP_AUDIENCE", "")
IAP_CERTS = "https://www.gstatic.com/iap/verify/public_key"
ADMIN_EMAILS = [e.strip().lower() for e in
                os.environ.get("ADMIN_EMAILS", "").split(",") if e.strip()]


def current_user() -> dict:
    if not IAP_AUDIENCE:
        # Fail closed. Verifying without an audience accepts a token minted for
        # ANY service behind IAP in this project.
        st.error("IAP_AUDIENCE is not set; refusing to authenticate.")
        st.stop()
    token = st.context.headers.get("x-goog-iap-jwt-assertion")
    if not token:
        st.error("403 - reach this service through IAP.")
        st.stop()
    try:
        claims = id_token.verify_token(token, g_requests.Request(),
                                       audience=IAP_AUDIENCE, certs_url=IAP_CERTS)
    except Exception:
        st.error("403 - invalid IAP assertion.")
        st.stop()
    email = (claims.get("email") or "").lower()
    # The IAP group binding is the gate; this is defence in depth, and it is the
    # check that still holds if somebody widens the group by accident.
    if ADMIN_EMAILS and email not in ADMIN_EMAILS:
        st.error("403 - Admins only")
        st.stop()
    return {"email": email, "sub": claims.get("sub")}
'''

REQUIREMENTS = '''streamlit==1.63.0
pandas==2.3.3
plotly==5.24.1
google-cloud-bigquery[pandas]==3.45.0
google-cloud-firestore==2.30.0
google-cloud-storage==3.13.1
google-cloud-dlp==3.39.0
google-auth==2.57.1
'''

DOCKERFILE = '''# syntax=docker/dockerfile:1.7
# Built from the deploy/ directory, not services/admin/, so shared/ ships with it:
#   docker build -f services/admin/Dockerfile -t admin .
FROM python:3.12-slim
RUN useradd --create-home --shell /bin/bash --uid 10001 app
WORKDIR /app
COPY services/admin/requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY --chown=app:app shared/ ./shared/
COPY --chown=app:app services/admin/ .
USER app
ENV PORT=8080 PYTHONUNBUFFERED=1
EXPOSE 8080
CMD ["streamlit", "run", "app.py", "--server.port=8080", "--server.address=0.0.0.0", "--server.headless=true"]
'''

for name, body in (("app.py", APP_PY), ("auth.py", AUTH_PY),
                   ("requirements.txt", REQUIREMENTS)):
    with open(name, "w") as f:
        f.write(body)
    print(f"wrote {name}")

# Written on its own line: the extractor pairs a filename with a var either from an
# open(...).write(VAR) call or from a ("name.ext", VAR) tuple - and the tuple form
# requires an extension. "Dockerfile" has none, so the tuple would be skipped and the
# file would silently not ship.
with open("Dockerfile", "w") as f:
    f.write(DOCKERFILE)
print("wrote Dockerfile")

print()
print("documind-admin is now its own image:")
print("  entrypoint : app.py -> admin_page(current_user())")
print("  identity   : auth.py verifies the IAP JWT signature, not a header")
print("  shared/    : pii.py + audit_log.py, the same modules the ingest worker uses")


In [ ]:
ALERTS_TF = '''
variable "pagerduty_key" {
  type      = string
  sensitive = true
}

resource "google_monitoring_notification_channel" "oncall" {
  # The Makefile passes `unset-in-dryrun` when nobody has a PagerDuty key; a channel with
  # a placeholder key is worse than none, so none: the policies below still exist and show
  # in the console, and a real key on a later apply creates the channel and wires it in.
  count        = var.pagerduty_key == "unset-in-dryrun" ? 0 : 1
  display_name = "DocuMind on-call"
  type         = "pagerduty"
  sensitive_labels {
    service_key = var.pagerduty_key
  }
}

# SLO: p95 /v1/query < 3s over 30-min rolling window
resource "google_monitoring_alert_policy" "api_latency" {
  display_name = "API p95 latency > 3s"
  combiner     = "OR"
  conditions {
    display_name = "p95 > 3s for 5 minutes"
    condition_threshold {
      filter     = "resource.type=\\"cloud_run_revision\\" AND resource.labels.service_name=\\"documind-api\\" AND metric.type=\\"run.googleapis.com/request_latencies\\""
      comparison = "COMPARISON_GT"
      threshold_value = 3000
      duration   = "300s"
      aggregations {
        alignment_period     = "60s"
        per_series_aligner   = "ALIGN_PERCENTILE_95"
        cross_series_reducer = "REDUCE_MEAN"
      }
    }
  }
  notification_channels = local.alert_channel_ids
  alert_strategy { auto_close = "1800s" }
}

# Unanswerable rate spike (product signal, not infra)
# The alert reads two log-based metrics rag-api's query log feeds and alerts on their ratio.
#
# The first version extracted rag-api's 0/1 unanswerable_flag into ONE distribution metric
# and asked Monitoring for ALIGN_MEAN over it. Monitoring refused, on the first live apply
# (the first live run, 6 September 2026): a mean is defined for numeric series, not distributions,
# and an alert filter has to name a resource type. Two counters and a ratio is what the API
# allows, and it is also the honest shape - a rate is a numerator over a denominator.
#
# Before that, NOTHING created the metric the alert read, so terraform applied cleanly, the
# policy showed green in the console, and it could never fire. An alert that cannot fire is
# worse than no alert: it is a promise someone is relying on. The flag is still emitted (0/1
# beside the boolean) because a log filter matches on it directly.
resource "google_logging_metric" "unanswerable" {
  name    = "documind/unanswerable"
  project = var.project_id
  filter  = <<EOT
    resource.type="cloud_run_revision"
    resource.labels.service_name="documind-api"
    (jsonPayload.event="query" OR jsonPayload.event="stream")
    jsonPayload.unanswerable_flag=1
  EOT
  metric_descriptor {
    metric_kind = "DELTA"
    value_type  = "INT64"
    unit        = "1"
    labels {
      key         = "tenant"
      value_type  = "STRING"
      description = "Tenant the question belonged to"
    }
  }
  label_extractors = {
    tenant = "EXTRACT(jsonPayload.tenant)"
  }
}

resource "google_logging_metric" "queries" {
  name    = "documind/queries"
  project = var.project_id
  filter  = <<EOT
    resource.type="cloud_run_revision"
    resource.labels.service_name="documind-api"
    (jsonPayload.event="query" OR jsonPayload.event="stream")
  EOT
  metric_descriptor {
    metric_kind = "DELTA"
    value_type  = "INT64"
    unit        = "1"
    labels {
      key         = "tenant"
      value_type  = "STRING"
      description = "Tenant the question belonged to"
    }
  }
  label_extractors = {
    tenant = "EXTRACT(jsonPayload.tenant)"
  }
}

resource "google_monitoring_alert_policy" "unanswerable_rate" {
  display_name = "Unanswerable rate > 20% for a tenant"
  combiner     = "OR"
  conditions {
    display_name = "unanswerable / queries > 0.20 for 30 minutes"
    condition_threshold {
      filter             = "resource.type=\\"cloud_run_revision\\" AND metric.type=\\"logging.googleapis.com/user/documind/unanswerable\\""
      denominator_filter = "resource.type=\\"cloud_run_revision\\" AND metric.type=\\"logging.googleapis.com/user/documind/queries\\""
      comparison         = "COMPARISON_GT"
      threshold_value    = 0.20
      duration           = "1800s"
      aggregations {
        alignment_period     = "300s"
        per_series_aligner   = "ALIGN_DELTA"
        cross_series_reducer = "REDUCE_SUM"
        group_by_fields      = ["metric.label.tenant"]
      }
      denominator_aggregations {
        alignment_period     = "300s"
        per_series_aligner   = "ALIGN_DELTA"
        cross_series_reducer = "REDUCE_SUM"
        group_by_fields      = ["metric.label.tenant"]
      }
    }
  }
  notification_channels = local.alert_channel_ids
  depends_on            = [google_logging_metric.unanswerable, google_logging_metric.queries]
}

# Cost control (10 September 2026): a GPU service left warm. instance_count is a gauge of a service's container
# instances, active or idle; a warm L4 instance is $1.42 an hour (11.1) and looks perfectly healthy for four weeks.
# Two hours above zero is the alarm - the nightly job in off.tf is the switch, this is the earlier warning, and
# make gpu-cap is the ceiling under both. A session runs two hours too, so the alarm fires near the end of one;
# that is the reminder, not a bug.
resource "google_monitoring_alert_policy" "gpu_left_warm" {
  for_each     = toset(["documind-slm", "documind-vllm"])
  display_name = "${each.key} left warm: instances > 0 for 2 hours"
  combiner     = "OR"
  conditions {
    display_name = "container instances > 0 for 2 hours"
    condition_threshold {
      filter          = "resource.type=\\"cloud_run_revision\\" AND resource.labels.service_name=\\"${each.key}\\" AND metric.type=\\"run.googleapis.com/container/instance_count\\""
      comparison      = "COMPARISON_GT"
      threshold_value = 0
      duration        = "7200s"
      aggregations {
        alignment_period     = "300s"
        per_series_aligner   = "ALIGN_MAX"
        cross_series_reducer = "REDUCE_SUM"
        group_by_fields      = ["resource.label.service_name"]
      }
    }
  }
  notification_channels = local.alert_channel_ids
  alert_strategy { auto_close = "1800s" }
  documentation {
    content   = "An L4 instance has been up for two hours. If a session is not running: make off PROJECT=<project> (or wait for the 23:00 IST job), then check with `gcloud run services describe ${each.key}`. A warm instance is Rs 86,904 a month."
    mime_type = "text/markdown"
  }
}

# The document lifecycle, observed (12 September 2026, deploy/INDEXING.md). The ingest worker logs one structured
# line per event - ingest_ok, ingest_superseded, ingest_reactivated, ingest_stale_event, ingest_failed - with the
# counts a reindex cost (reused, embedded, retired); the nightly job ends on reconcile_done with the drift it
# measured. Three metrics read them off the log, and three policies turn them into a pager: a failure within ten
# minutes, drift that stays above zero for two nights, the job itself failing. Before these, every one of those
# lines was a log search somebody had to remember to run.
resource "google_logging_metric" "ingest_events" {
  name    = "documind/ingest_events"
  project = var.project_id
  filter  = <<EOT
    resource.type="cloud_run_revision"
    resource.labels.service_name="documind-ingest"
    jsonPayload.event=~"^ingest_(ok|superseded|reactivated|stale_event|failed)$"
  EOT
  metric_descriptor {
    metric_kind = "DELTA"
    value_type  = "INT64"
    unit        = "1"
    labels {
      key         = "event"
      value_type  = "STRING"
      description = "ingest_ok | ingest_superseded | ingest_reactivated | ingest_stale_event | ingest_failed"
    }
    labels {
      key         = "tenant"
      value_type  = "STRING"
      description = "Tenant the object belonged to"
    }
  }
  label_extractors = {
    event  = "EXTRACT(jsonPayload.event)"
    tenant = "EXTRACT(jsonPayload.tenant)"
  }
}

# What a reindex costs, as a distribution of the embedded count per ingest_ok: the number the carry-over keeps
# small. reused rides beside it; a chart of the two is "pay for what changed" made visible.
resource "google_logging_metric" "ingest_embedded" {
  name    = "documind/ingest_embedded"
  project = var.project_id
  filter  = <<EOT
    resource.type="cloud_run_revision"
    resource.labels.service_name="documind-ingest"
    jsonPayload.event="ingest_ok"
  EOT
  metric_descriptor {
    metric_kind = "DELTA"
    value_type  = "DISTRIBUTION"
    unit        = "1"
    labels {
      key         = "tenant"
      value_type  = "STRING"
      description = "Tenant the object belonged to"
    }
  }
  value_extractor = "EXTRACT(jsonPayload.embedded)"
  label_extractors = {
    tenant = "EXTRACT(jsonPayload.tenant)"
  }
  bucket_options {
    explicit_buckets {
      bounds = [0, 1, 2, 5, 10, 25, 50, 100, 250, 500, 1000, 2500]
    }
  }
}

resource "google_logging_metric" "ingest_reused" {
  name    = "documind/ingest_reused"
  project = var.project_id
  filter  = <<EOT
    resource.type="cloud_run_revision"
    resource.labels.service_name="documind-ingest"
    jsonPayload.event="ingest_ok"
  EOT
  metric_descriptor {
    metric_kind = "DELTA"
    value_type  = "DISTRIBUTION"
    unit        = "1"
    labels {
      key         = "tenant"
      value_type  = "STRING"
      description = "Tenant the object belonged to"
    }
  }
  value_extractor = "EXTRACT(jsonPayload.reused)"
  label_extractors = {
    tenant = "EXTRACT(jsonPayload.tenant)"
  }
  bucket_options {
    explicit_buckets {
      bounds = [0, 1, 2, 5, 10, 25, 50, 100, 250, 500, 1000, 2500]
    }
  }
}

# The night's number. reconcile.py prints reconcile_done with `drift` from the job (resource.type cloud_run_job);
# a metadata-only change is not drift, a lost event or a deleted object is.
resource "google_logging_metric" "reconcile_drift" {
  name    = "documind/reconcile_drift"
  project = var.project_id
  filter  = <<EOT
    resource.type="cloud_run_job"
    resource.labels.job_name="documind-reconcile"
    jsonPayload.event="reconcile_done"
  EOT
  metric_descriptor {
    metric_kind = "DELTA"
    value_type  = "DISTRIBUTION"
    unit        = "1"
  }
  value_extractor = "EXTRACT(jsonPayload.drift)"
  bucket_options {
    explicit_buckets {
      bounds = [0, 1, 2, 5, 10, 50, 100]
    }
  }
}

resource "google_monitoring_alert_policy" "ingest_failed" {
  display_name = "Ingest failed"
  combiner     = "OR"
  conditions {
    display_name = "an ingest_failed line in the last ten minutes"
    condition_threshold {
      filter          = "resource.type=\\"cloud_run_revision\\" AND metric.type=\\"logging.googleapis.com/user/documind/ingest_events\\" AND metric.label.event=\\"ingest_failed\\""
      comparison      = "COMPARISON_GT"
      threshold_value = 0
      duration        = "0s"
      aggregations {
        alignment_period     = "600s"
        per_series_aligner   = "ALIGN_DELTA"
        cross_series_reducer = "REDUCE_SUM"
        group_by_fields      = ["metric.label.tenant"]
      }
    }
  }
  notification_channels = local.alert_channel_ids
  alert_strategy { auto_close = "1800s" }
  documentation {
    content   = "A document failed to index (the claim was released, the message is retrying towards the DLQ). Read the line: gcloud logging read 'jsonPayload.event=\\"ingest_failed\\"' --limit 5; then make dlq. A poison object is ingest_poison, not this."
    mime_type = "text/markdown"
  }
  depends_on = [google_logging_metric.ingest_events]
}

# Two nights, not one: a single night's drift is a lost event the reconcile just repaired; drift that is still
# above zero after the next run is a lane nobody is reconciling (the apply failing, the job not scheduled).
resource "google_monitoring_alert_policy" "reconcile_drift" {
  count        = var.reconcile_job ? 1 : 0
  display_name = "Ledger drift above zero for two nights"
  combiner     = "OR"
  conditions {
    display_name = "reconcile_done.drift > 0 for 24 hours (two nightly runs)"
    condition_threshold {
      filter          = "resource.type=\\"cloud_run_job\\" AND metric.type=\\"logging.googleapis.com/user/documind/reconcile_drift\\""
      comparison      = "COMPARISON_GT"
      threshold_value = 0
      duration        = "86400s"
      aggregations {
        alignment_period     = "86400s"
        per_series_aligner   = "ALIGN_PERCENTILE_99"
        cross_series_reducer = "REDUCE_MAX"
      }
    }
  }
  notification_channels = local.alert_channel_ids
  documentation {
    content   = "The nightly reconcile found the ledger out of step with the uploads bucket two runs in a row. make reconcile PROJECT=<project> prints the plan; make reconcile APPLY=1 acts; make sources TENANT= shows the ledger."
    mime_type = "text/markdown"
  }
  depends_on = [google_logging_metric.reconcile_drift]
}

resource "google_monitoring_alert_policy" "reconcile_failed" {
  count        = var.reconcile_job ? 1 : 0
  display_name = "Nightly reconcile failed"
  combiner     = "OR"
  conditions {
    display_name = "a documind-reconcile task attempt failed"
    condition_threshold {
      filter          = "resource.type=\\"cloud_run_job\\" AND resource.labels.job_name=\\"documind-reconcile\\" AND metric.type=\\"run.googleapis.com/job/completed_task_attempt_count\\" AND metric.labels.result=\\"failed\\""
      comparison      = "COMPARISON_GT"
      threshold_value = 0
      duration        = "0s"
      aggregations {
        alignment_period     = "3600s"
        per_series_aligner   = "ALIGN_DELTA"
        cross_series_reducer = "REDUCE_SUM"
      }
    }
  }
  notification_channels = local.alert_channel_ids
  alert_strategy { auto_close = "86400s" }
  documentation {
    content   = "documind-reconcile did not finish. gcloud run jobs executions list --job documind-reconcile, then the execution's logs; run it by hand with gcloud run jobs execute documind-reconcile --region <region>."
    mime_type = "text/markdown"
  }
}

# The e-mail channel. PagerDuty is the on-call when pagerduty_key is set; the admins' addresses are the on-call otherwise
# (make up passes ALERT_EMAILS, derived from ADMIN_EMAILS unless that is still the placeholder), and every policy
# in this file notifies both channels when both exist.
variable "alert_emails" {
  type    = list(string)
  default = []
}

resource "google_monitoring_notification_channel" "email" {
  for_each     = toset(var.alert_emails)
  display_name = "DocuMind admin ${each.key}"
  type         = "email"
  labels       = {
    email_address = each.key
  }
}

locals {
  alert_channel_ids = concat(google_monitoring_notification_channel.oncall[*].id, [for c in google_monitoring_notification_channel.email : c.id])
}
'''
with open('alerts.tf', 'w') as f: f.write(ALERTS_TF)
print('alerts.tf written')
print()
print('Two policies: infra SLO (p95 latency) + product signal (unanswerable rate).')
print('Product signal catches retrieval regressions that would pass every infra check.')


In [ ]:
DEPLOY = '''
# documind-admin is its OWN service and its OWN image. It used to deploy ui:$GIT_SHA,
# which is built from services/frontend/ only - so services/admin/*.py was in no
# image at all and this service came up as a second copy of the chat UI. Deployed,
# healthy, reachable, and not the code you wrote.
#
# Build context is deploy/, not services/admin/, so shared/ (pii.py, audit_log.py)
# ships with it - the same modules the ingest worker scans with.
gcloud builds submit \\
  --tag=us-central1-docker.pkg.dev/$PROJECT/documind/admin:$GIT_SHA \\
  --file=services/admin/Dockerfile .

gcloud run deploy documind-admin \\
  --image=us-central1-docker.pkg.dev/$PROJECT/documind/admin:$GIT_SHA \\
  --region=us-central1 --no-allow-unauthenticated \\
  --ingress=internal-and-cloud-load-balancing \\
  --service-account=documind-admin-sa@$PROJECT.iam.gserviceaccount.com \\
  --set-env-vars=^:^GOOGLE_CLOUD_PROJECT=$PROJECT:ADMIN_EMAILS=alice@documind.ai,bob@documind.ai:AUDIT_BUCKET=$PROJECT-audit:IAP_AUDIENCE=$IAP_AUDIENCE \\
  --set-secrets="COOKIE_SECRET=cookie-secret:latest" \\
  --session-affinity --cpu-boost

# IAP group restricted to admin-group@documind.ai
gcloud beta run services update documind-admin --region=us-central1 --iap
gcloud beta iap web add-iam-policy-binding \\
  --resource-type=cloud-run --service=documind-admin --region=us-central1 \\
  --member="group:admin-group@documind.ai" \\
  --role="roles/iap.httpsResourceAccessor"

# Admin SA needs BQ data viewer + Firestore user
gcloud projects add-iam-policy-binding $PROJECT \\
  --member="serviceAccount:documind-admin-sa@$PROJECT.iam.gserviceaccount.com" \\
  --role="roles/bigquery.dataViewer"
gcloud projects add-iam-policy-binding $PROJECT \\
  --member="serviceAccount:documind-admin-sa@$PROJECT.iam.gserviceaccount.com" \\
  --role="roles/bigquery.jobUser"
'''
print(DEPLOY)


In [ ]:
SMOKE = '''
# 1. Sink is writing
bq query --use_legacy_sql=false \\
  "SELECT COUNT(*) FROM documind_observability.run_googleapis_com_stdout \\
   WHERE timestamp > TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 1 HOUR)"
# -> positive integer = sink healthy

# 2. DLP finds PII in a test doc
python -c "from dlp import inspect_and_log; print(inspect_and_log(\\
  'test_chunk', 'tenant-acme', 'Contact raj@example.in PAN ABCDE1234F'))"
# -> {"has_pii": true, "types": ["EMAIL_ADDRESS","INDIA_PAN_INDIVIDUAL"]}

# 3. Audit emit writes to GCS
python -c "from audit import emit; print(emit(\\
  \"doc.upload\", {\"email\":\"alice@acme.in\",\"tenant_id\":\"tenant-acme\"},\\
  {\"type\":\"doc\",\"id\":\"doc_9f23\"}))"
gsutil ls gs://$PROJECT-audit/$(date +%Y)/$(date +%m)/$(date +%d)/tenant-acme/

# 4. Non-admin user cannot see admin tab
# Open documind-admin Cloud Run URL as a non-admin user -> IAP 403.

# 5. Alert fires on synthetic latency spike
hey -n 200 -c 20 -m POST -H "Authorization: Bearer $TOK" \\
  https://documind-api-xxx.run.app/v1/query
# Wait 6 min, PagerDuty page arrives if p95 > 3000ms
'''
print(SMOKE)
print()
print('WHAT 12.4 (Streamlit UI) INHERITS FROM THIS LESSON:')
print('  - admin_dashboard.py already implements the Admin tab')
print('  - audit.emit() called on doc.upload / doc.delete / query.export in the UI flows')
print('  - dlp.redact() applied before any prompt routed to LiteLLM non-India backend')
print('  - Admins access documind-admin Cloud Run service via IAP group membership')
